# 06 Trees - Homework - Ahmed Elsherif

### Setup

In [64]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import export_text

### Dataset

In [65]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-11-01 18:24:09--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv.2’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.04s   

2026-11-01 18:24:09 (16.6 MB/s) - ‘car_fuel_efficiency_2026.csv.2’ saved [635180/635180]



In [66]:
df = pd.read_csv("car_fuel_efficiency_2026.csv")

### Preparing the dataset

Preparation:

* Fill missing values with zeros.
* Do the split with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
```

* Use `DictVectorizer(sparse=True)` to turn the dataframes into matrices.


In [67]:
df = df.fillna(0)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [68]:
y_train, y_val, y_test = df_train['fuel_efficiency_mpg'], df_val['fuel_efficiency_mpg'], df_test['fuel_efficiency_mpg']
X_train = df_train.drop(columns=['fuel_efficiency_mpg'])
X_val = df_val.drop(columns=['fuel_efficiency_mpg'])
X_test = df_test.drop(columns=['fuel_efficiency_mpg'])


In [69]:
dicts_train = X_train.to_dict(orient='records')
dicts_val = X_val.to_dict(orient='records')
dicts_test = X_test.to_dict(orient='records')

dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(dicts_train)
X_val = dv.transform(dicts_val)
X_test = dv.transform(dicts_test)



## Question 1

Let's train a decision tree regressor to predict the `fuel_efficiency_mpg` variable.

* Train a model with `max_depth=1`.


Which original feature is used for splitting the data? If you use
`DictVectorizer`, map a one-hot feature such as `origin=USA` back to its
original column name.

* **Answer:** `'model_year'`

In [70]:
dt_regressor = DecisionTreeRegressor(max_depth=1)
dt_regressor.fit(X_train, y_train)
print(export_text(dt_regressor, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



## Question 2

Train a random forest regressor with these parameters:

* `n_estimators=10`
* `random_state=1`
* `n_jobs=-1` (optional - to make training faster)


What's the RMSE of this model on the validation data?

* **Answer:** `1.837`

In [71]:
rf_regressor = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf_regressor.fit(X_train, y_train)
y_pred = rf_regressor.predict(X_val)
val_rmse = root_mean_squared_error(y_val, y_pred)
print("Validation RMSE:", round(val_rmse,3))

Validation RMSE: 1.837


## Question 3

Now let's experiment with the `n_estimators` parameter

* Try these values of this parameter: `[10, 50, 100, 150]`.
* Set `random_state` to `1`.
* Evaluate the model on the validation dataset.


Which value gives the lowest validation RMSE? Consider 3 decimal places for
calculating the answer.

- **Answer:** `100`

In [72]:
n_estimators_list = [10, 50, 100, 150]

results = []

for n in n_estimators_list:
  rf_regressor = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
  rf_regressor.fit(X_train, y_train)
  y_pred = rf_regressor.predict(X_val)
  val_rmse = round(root_mean_squared_error(y_val, y_pred), 3)
  results.append({
      "n_estimators":n,
      "validation_rmse": val_rmse
  })

results_df = pd.DataFrame(results).sort_values(by="validation_rmse", ascending=True)



best_result = results_df.loc[
    results_df["validation_rmse"].idxmin()
]

print("Lowest RMSE:\n", best_result)
print("\n")
results_df


Lowest RMSE:
 n_estimators       100.000
validation_rmse      1.768
Name: 2, dtype: float64




,n_estimators,validation_rmse
2,100,1.768
3,150,1.769
1,50,1.772
0,10,1.837


## Question 4

Let's select the best `max_depth`:

* Try different values of `max_depth`: `[10, 15, 20, 25]`
* For each of these values,
  * try different values of `n_estimators` from `[10, 50, 100, 150]`
  * calculate the mean RMSE
* Fix the random seed: `random_state=1`


What's the best `max_depth`, using the mean RMSE?

* **Answer:** `10`

In [73]:
max_depth_list = [10, 15, 20, 25]
n_estimators_list = [10, 50, 100, 150]


results = []

for max_depth in max_depth_list:
    for n in n_estimators_list:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=max_depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)

        results.append({
            "max_depth": max_depth,
            "n_estimators": n,
            "rmse": root_mean_squared_error(y_val, y_pred)
        })

results_df = pd.DataFrame(results)

results_df.groupby("max_depth")["rmse"].mean().sort_values()

,rmse
max_depth,
10,1.755405
15,1.783953
25,1.786317
20,1.788242


# Question 5

We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split.
When doing it, we can calculate "gain" - the reduction in impurity before and after the split.
This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the
[`feature_importances_`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor.feature_importances_)
field.

For this homework question, we'll find the most important feature:

* Train the model with these parameters:
  * `n_estimators=10`,
  * `max_depth=20`,
  * `random_state=1`,
  * `n_jobs=-1` (optional)
* Get the feature importance information from this model


What's the most important feature (among these 4)?

* **Answer:** `vehicle_weight`

In [74]:
rf_model = RandomForestRegressor(n_estimators=10,max_depth=20,random_state=1,n_jobs=-1)

rf_model.fit(X_train, y_train)

feature_importance_df = pd.DataFrame({
    "feature": dv.get_feature_names_out(),
    "importance": rf_model.feature_importances_
})

feature_importance_df[
    feature_importance_df["feature"].isin([
        "vehicle_weight",
        "horsepower",
        "acceleration",
        "engine_displacement"
    ])
].sort_values(by="importance", ascending=False)

,feature,importance
15,vehicle_weight,0.203879
8,horsepower,0.078017
4,engine_displacement,0.061005
0,acceleration,0.055201


## Question 6

Now let's train an XGBoost model! For this question, we'll tune the `eta` parameter:

* Install XGBoost
* Create DMatrix for train and validation
* Create a watchlist
* Train a model with these parameters for 100 rounds:

```
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
```

Now change `eta` from `0.3` to `0.1`.

Which eta leads to the best RMSE score on the validation dataset?


* **Answer:** `0.1`

In [75]:
!pip install xgboost

In [76]:
import xgboost as xgb

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval, 'val')]


xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

# eta = 0.3
model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist
)

y_pred = model.predict(dval)
rmse_3 = root_mean_squared_error(y_val, y_pred)


# eta = 0.1
xgb_params["eta"] = 0.1

model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist
)

y_pred = model.predict(dval)
rmse_1 = root_mean_squared_error(y_val, y_pred)

print(f"eta=0.3: {rmse_3:.5f}")
print(f"eta=0.1: {rmse_1:.5f}")

[0]	train-rmse:2.40653	val-rmse:2.46782
[1]	train-rmse:2.07098	val-rmse:2.15784
[2]	train-rmse:1.86454	val-rmse:1.97398
[3]	train-rmse:1.73360	val-rmse:1.86662
[4]	train-rmse:1.65059	val-rmse:1.80312
[5]	train-rmse:1.59328	val-rmse:1.76216
[6]	train-rmse:1.55435	val-rmse:1.74392
[7]	train-rmse:1.52677	val-rmse:1.73976
[8]	train-rmse:1.50265	val-rmse:1.73478
[9]	train-rmse:1.48761	val-rmse:1.73282
[10]	train-rmse:1.47125	val-rmse:1.73205
[11]	train-rmse:1.45902	val-rmse:1.73227
[12]	train-rmse:1.44829	val-rmse:1.73503
[13]	train-rmse:1.43604	val-rmse:1.73466
[14]	train-rmse:1.42972	val-rmse:1.73448
[15]	train-rmse:1.41908	val-rmse:1.73512
[16]	train-rmse:1.41206	val-rmse:1.73297
[17]	train-rmse:1.40940	val-rmse:1.73517
[18]	train-rmse:1.40278	val-rmse:1.73584
[19]	train-rmse:1.39637	val-rmse:1.73664
[20]	train-rmse:1.39368	val-rmse:1.73766
[21]	train-rmse:1.39254	val-rmse:1.73701
[22]	train-rmse:1.37639	val-rmse:1.74216
[23]	train-rmse:1.36470	val-rmse:1.74363
[24]	train-rmse:1.36071	va